FIX CODE MODEL SIMPLE USE IT PLEASE DONT EDIT ANYMORE

In [16]:
from model_system import run_hybrid_summary

results = run_hybrid_summary(desired_array_size=120000, P_ref=110, t_TES_hours=14)
print(f"Capacity Factor : {results.get('CF_hybrid')*100:.2f} %")
print(f"PPA Price MUSD  : {results.get('PPA_USD_per_MWh'):.2f}")

Capacity Factor : 69.96 %
PPA Price MUSD  : 40.41


In [ ]:
from models.pv_model import run_pvsam
from viz.plots import plot_week
from capex_opex.excel_capex_opex import calc_capex_opex
from pathlib import Path
from models.csp_model import run_basic_mspt
import sys
sys.path.append(".")
from models.dispatch_opt import optimize_dispatch, export_dispatch_to_csv,calculate_firm_bonus

#===============================================================================================
# Run the PV model (PySAM)
#===============================================================================================
df_pv, pv_summary, pv_obj, annual_energy_kwh, land_area_m2, name_plate_kwdc = run_pvsam(
    weather_path="input_data/tmy_38.730_-3.450_2005_2023.epw",
    module_name="SunPower SPR-E19-315",
    inverter_name="Sungrow Power Supply Co - Ltd : SC2500U [550V]",
    cec_module_csv="equipment_database/CEC Modules.csv",
    cec_inverter_csv="equipment_database/CEC Inverters.csv",
    system_kwargs=dict(
        inverter_count=30,
        subarray1_modules_per_string=21,
        subarray1_nstrings=15_354,
        subarray1_track_mode=0,
        subarray1_tilt=20.0,
        subarray1_azimuth=180.0,
        subarray1_gcr=0.4,
        mppt_low_inverter=800.0,
        mppt_hi_inverter=1500.0,
        albedo=[0.2]*12,
        soiling=[5]*12,
    )
)

print("\n--- PV Simulation Summary ---")
for k, v in pv_summary.items():
    print(f"{k:25s}: {v:,.2f}") 

#===============================================================================================
# Run the CSP model (PySAM)
#===============================================================================================

weather_path = "input_data/tmy_38.730_-3.450_2005_2023.epw"

df_csp, csp_summary, mspt = run_basic_mspt(
    weather_path,
    P_ref=50,
    design_eff=0.412,
    solarm=2.4,
    tshours=0,                  # have to 0 to run in PySAM, put the number in dispatch optimization   
    dni_des=950.0,
    T_hot=565.0,
    T_cold=290.0,
    is_dispatch=0,              # no optimization (fast)
    time_steps_per_hour=1,      # coarsest sim grid
    disp_steps_per_hour=1,      # coarsest dispatch grid
)

csv_path = "output_data/csp_hourly_results.csv"
df_pv.to_csv("output_data/pv_hourly_results.csv", index=False)
df_csp.to_csv(csv_path, index=False)
print("\n--- CSP Simulation Summary ---")
for k, v in csp_summary.items():
    print(f"{k:25s}: {v:,.2f}") 

#===============================================================================================
# Dispatch Optimization
#===============================================================================================
q_csp = df_csp['P_rec_MWt_calculated'].tolist()  
e_pv  = (df_pv['AC_kWh'] / 1000).tolist() 
t_TES_hours = 10        # hours of thermal energy storage capacity
x_price = 1             # EUR/kWh electricity price
max_to_grid = 100       # MWe limit
res, model = optimize_dispatch(
    q_csp, 
    e_pv, 
    x_price=x_price,
    max_to_grid=max_to_grid,                                        
    PB_e_max=csp_summary.get("NetCapacity_MWe"),
    E_cap=csp_summary.get("NetCapacity_MWe")*t_TES_hours,
    )  

bonus = calculate_firm_bonus(res, base_price=50.0, bonus_rate=0.2, margin=0.05, min_hours=3)

export_dispatch_to_csv(res, "output_data")

print("\n--- Dispatch Optimization Revenue ---")
print(f"Revenue : {res['revenue_total_eur']:,.2f}")
print(f"💰 Bonus revenue: {bonus['bonus_revenue']:,.2f}")
print(f"💰 Total revenue (with bonus): {bonus['total_revenue_with_bonus']:,.2f}")
print(f"✅ Firm operation hours: {bonus['firm_hours']}")

#===============================================================================================
# CAPEX AND OPEX CALCULATION USING EXCEL MODEL
#===============================================================================================
excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")

inputs = {
    "PB Installed Capacity (Gross)":            csp_summary.get("NetCapacity_MWe"),                   # MW
    "Solar Field Aperture Area (Mirror Area)":  csp_summary.get("Solar_Field_Area_m2"),               # m2
    "Thermal Energy Storage Capacity ":         csp_summary.get("Tes_Capacity_MWh"),                  # MWh-t
    "Receiver Power (Max Rated)":               0,                                                    # MW 0 if Parabolic Trough
    "Tower Height (w/o Receiver)":              csp_summary.get("Tower_Height_m"),                    # m
    "Electric Heater Thermal Power (Max Rated)":csp_summary.get("Electric_Heater_Power_MWt"),         # MWt
    "Land Area CSP":                            csp_summary.get("Land_Area_acre") * 4046.86,          # m2
    "PV Installed Capacity":                    name_plate_kwdc *1000,                                # Wdc 
    "Battery Pack Power (Max Rated)":           0,                                                    # MW bess_energy_mwh or 
    "Battery Pack Capacity":                    0,                                                      # MWh-e bess_energy_mwh or 
    "Battery Annual Generation (for OPEX)":     0,                                                    # GWh/y
    "Land Area PV":                             land_area_m2,                                           # m2
    "CSP Annual Generation (for OPEX)":         csp_summary.get("Annual_kWh") / 1000,                 # GWh/y
    "Distance to Grid ":                        0,
    "Distance to Road":                         0,
    "Distance to Gas":                          0,
    "Distance to Water":                        0,
    "Other Component Size":                     0,
}

overrides = {
    # If tower technology: SET BOP Reference to 74.8 MUSD or If PT technology: SET BOP Reference Cost to 90.2 MUSD
    "BOP": 74.8,          # MUSD (replaces CAPEX!D27)
    # If tower technology: SET Solar Field Reference Cost to 125 MUSD or If PT technology: SET Solar Field Reference Cost to 115 MUSD
    "SF_aperture": 115,  # MUSD (replaces CAPEX!D28)
    # pv_modules_ref_cost_musd=0.33,# $/Wdc
    "PV_modules": 0.3,   # MUSD (replaces CAPEX!D41)
    #Fixed Trackers	0.01 ; Single Axis Tracking	0.0175
    "pv_fixed_opex_coeff": 1,
}

capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
    excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
)
print("\n--- CAPEX AND OPEX Simulation Summary ---")
print(f"Total CAPEX: {capex_musd:,.2f} MUSD")
print(f"Total OPEX : {opex_musd:,.2f} MUSD/year")
#plot_week(df_all, bess_energy_mwh=20.0, week_index=0)

# Parameters
debt = 0.8
equity = 0.2
debt_rate = 0.05
equity_rate = 0.08
tax_rate = 0.25
lifetime_years = 30
construction_years = 2

# Financial Calculation to find PPA
wacc = (debt * debt_rate * (1 - tax_rate)) + (equity * equity_rate)
sigma_capex = sum((capex_musd*10**6) / (construction_years*(1 + wacc)**t) for t in range(0, construction_years - 1))
sigma_opex = sum(1 / (1 + wacc)**t for t in range(1,lifetime_years + construction_years -1))
A = sigma_capex / sigma_opex
PPA = (A+opex_musd*10**6) / bonus['total_revenue_with_bonus']

print(f"Present Value Factor (S): {sigma_capex:,.2f}")
print(f"A value: {A:,.2f}")
print(f"PPA value: {PPA:,.2f} USD/MWh or {PPA/1000:,.2f} USD/kWh")





--- PV Simulation Summary ---
Annual_kWh               : 169,969,768.01
CapacityFactor_AC_%      : 22.76
NamePlate_kWdc           : 102,004.44
LandArea_m2              : 39,278,665.67

--- CSP Simulation Summary ---
NetCapacity_MWe          : 100.00
Annual_kWh               : 203,887,351.11
Tower_Height_m           : 194.23
Land_Area_acre           : 2,417.62
Solar_Field_Area_m2      : 1,529,073.51
Receiver_Design_MWt      : 728.16
CapacityFactor_%         : 27.48
SolarMultiple            : 3.00

--- TES & Heater ---
Electric_Heater_Power_MWt : 250.00
Tes_Capacity_MWh : 4,000.00
TES_Hours : 16.0

--- CAPACITY FACTOR ---
PV CSP Cap Factor : 0.67
CSP Cap Factor : 0.48

--- Dispatch Optimization Results ---
Revenue (Σ α*E[t]): 790,382.71
💰 Bonus revenue (Σ α*E[t]): 147,780.89
💰 Total revenue (with bonus) (Σ α*E[t]): 938,163.60
✅ Firm operation hours: 5400

--- CAPEX/OPEX Input Parameters ---
PB Installed Capacity (Gross)                : 100,000.000
Solar Field Aperture Area (Mirror Area

GENETIC ALGORITHM

In [ ]:
from __future__ import annotations
import random, math
from functools import lru_cache
from pathlib import Path
from typing import Tuple
import numpy as np

from deap import base, creator, tools, algorithms

# === Your imports (as in your script) ===
from models.pv_model import run_pvsam
from viz.plots import plot_week  # (unused in GA, but keep if you need later)
from capex_opex.excel_capex_opex import calc_capex_opex
from models.csp_model import run_basic_mspt
from models.dispatch_opt import optimize_dispatch, export_dispatch_to_csv, calculate_firm_bonus

# ----------------------------
# CONSTANTS from your script
# ----------------------------
excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")

# Fixed parameters (can also be GA variables later if you want)
dc_to_ac_ratio = 1.2           # PV DC/AC
solarm          = 2.4          # DNI kWh/m2/day (kept fixed per your message)
max_to_grid     = 100.0        # MWe limit to grid
x_price         = 1.0          # EUR/kWh electricity price

# Finance
debt, equity        = 0.8, 0.2
debt_rate, eq_rate  = 0.05, 0.08
tax_rate            = 0.25
lifetime_years      = 30
construction_years  = 2

# Site distances (kept fixed)
dist_to_grid = 0
dist_to_road = 0
dist_to_gas  = 0
dist_to_water= 0
other_comp   = 0

# CAPEX overrides (as in your code)
overrides = {
    "BOP": 74.8,         # MUSD
    "SF_aperture": 115,  # MUSD
    "PV_modules": 0.3,   # MUSD
    "pv_fixed_opex_coeff": 1,
}

# ----------------------------
# GA decision variables & BOUNDS
# x = [ desired_array_size_kWdc,   P_ref_MWe,   t_TES_hours ]
# ----------------------------
BOUNDS = [
    (80_000.0, 120_000.0),   # PV DC size (kWdc)  <-- adjust if you want wider/narrower
    (40.0,     100.0),       # CSP net MWe
    ( 8,      12.0),       # TES hours
]

# Step sizes / rounding rules (keeps cache effective & realistic)
def repair_and_round(x):
    # clip to bounds
    xr = [None]*3
    for i, (lo, hi) in enumerate(BOUNDS):
        xr[i] = min(max(x[i], lo), hi)
    # round for caching & practical sizing
    xr[0] = round(xr[0] / 5000.0) * 5000.0      # PV size in 5 MW steps
    xr[1] = round(xr[1])                        # CSP in 1 MW steps
    xr[2] = round(xr[2])                        # TES in 1 h steps
    return xr

# ----------------------------
# Objective helper (cache)
# ----------------------------
@lru_cache(maxsize=256)
def evaluate_tuple(pv_kwdc: float, P_ref: float, t_TES_hours: float) -> float:
    """
    Returns PPA (USD/MWh). On failure returns a large penalty value.
    Cache key uses rounded inputs (we round upstream).
    """
    try:
        # --- PV ---
        df_pv, pv_summary, pv_obj, annual_energy_kwh, land_area_m2, name_plate_kwdc = run_pvsam(
            system_kwargs=dict(
                desired_array_size=pv_kwdc,      # kWdc (your updated API)
                dc_to_ac_ratio=dc_to_ac_ratio,
            )
        )

        # --- CSP (fast tower template) ---
        df_csp, csp_summary, mspt = run_basic_mspt(
            P_ref=P_ref,
            solarm=solarm,
        )

        # Series for dispatch
        q_csp = df_csp['P_rec_MWt_calculated'].tolist()
        e_pv  = (df_pv['AC_kWh'] / 1000.0).tolist()  # MWe per hour

        # TES capacity in MWht
        PB_e_max = float(csp_summary.get("NetCapacity_MWe"))
        E_cap    = PB_e_max * float(t_TES_hours)

        # --- Dispatch ---
        res, model = optimize_dispatch(
            q_csp,
            e_pv,
            x_price=x_price,
            max_to_grid=max_to_grid,
            PB_e_max=PB_e_max,
            E_cap=E_cap,
        )

        # --- Firmness bonus ---
        bonus = calculate_firm_bonus(res, base_price=50.0, bonus_rate=0.2, margin=0.05, min_hours=3)

        # --- CAPEX/OPEX Excel ---
        inputs = {
            "PB Installed Capacity (Gross)":            PB_e_max,                                         # MW
            "Solar Field Aperture Area (Mirror Area)":  csp_summary.get("Solar_Field_Area_m2"),           # m2
            "Thermal Energy Storage Capacity ":         csp_summary.get("Tes_Capacity_MWh"),              # MWht
            "Receiver Power (Max Rated)":               0,                                                # MW (0 for PT)
            "Tower Height (w/o Receiver)":              csp_summary.get("Tower_Height_m"),                # m
            "Electric Heater Thermal Power (Max Rated)":csp_summary.get("Electric_Heater_Power_MWt"),     # MWt
            "Land Area CSP":                            csp_summary.get("Land_Area_acre") * 4046.86,      # m2
            "PV Installed Capacity":                    name_plate_kwdc * 1000.0,                         # Wdc
            "Battery Pack Power (Max Rated)":           0,                                                # MW
            "Battery Pack Capacity":                    0,                                                # MWhe
            "Battery Annual Generation (for OPEX)":     0,                                                # GWh/y
            "Land Area PV":                             land_area_m2,                                     # m2
            "CSP Annual Generation (for OPEX)":         csp_summary.get("Annual_kWh") / 1000.0,           # GWh/y
            "Distance to Grid ":                        dist_to_grid,
            "Distance to Road":                         dist_to_road,
            "Distance to Gas":                          dist_to_gas,
            "Distance to Water":                        dist_to_water,
            "Other Component Size":                     other_comp,
        }

        capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
            excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
        )

        # --- Finance (your formulae) ---
        wacc = float((debt * debt_rate * (1 - tax_rate)) + (equity * eq_rate))

        # Present value factor for CAPEX during construction
        capex_musd = float(capex_musd)  # ensure plain float
        opex_musd  = float(opex_musd)   # ensure plain float

        sigma_capex = 0.0
        for t in range(0, construction_years - 1):
            sigma_capex += (capex_musd * 1e6) / (construction_years * ((1.0 + wacc) ** float(t)))

        sigma_opex = 0.0
        for t in range(1, lifetime_years + construction_years - 1):
            sigma_opex += 1.0 / ((1.0 + wacc) ** float(t))

        sigma_opex = max(sigma_opex, 1e-9)            # protect
        A = float(sigma_capex / sigma_opex)

        # Total revenue incl. bonus from dispatch (force real & positive)
        total_revenue = float(bonus['total_revenue_with_bonus'])
        if not np.isfinite(total_revenue) or total_revenue <= 0.0:
            return 1e12  # penalize invalid economics

        # PPA (force real scalar)
        PPA = float((A + opex_musd * 1e6) / total_revenue)

        # Soft constraint: do not oversize PB relative to max_to_grid too much
        over_pb = max(0.0, PB_e_max - max_to_grid)
        penalty = 5e5 * over_pb

        # Final: force real scalar return (even if any upstream tried to go complex)
        return float(np.real(PPA + penalty))


    except Exception as e:
        # Any failure -> big penalty to steer GA away
        return 1e12

# ----------------------------
# Penalized objective for DEAP
# ----------------------------
def objective(ind):
    x = repair_and_round(ind)
    ppa = evaluate_tuple(x[0], x[1], x[2])
    return (ppa,)

# ----------------------------
# GA scaffolding
# ----------------------------
random.seed(13)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
# init within bounds
toolbox.register("attr_pv",  random.uniform, BOUNDS[0][0], BOUNDS[0][1])
toolbox.register("attr_csp", random.uniform, BOUNDS[1][0], BOUNDS[1][1])
toolbox.register("attr_tes", random.uniform, BOUNDS[2][0], BOUNDS[2][1])

def init_individual():
    return creator.Individual([toolbox.attr_pv(), toolbox.attr_csp(), toolbox.attr_tes()])

toolbox.register("individual", init_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", objective)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutPolynomialBounded,
                 eta=20.0,
                 low=[b[0] for b in BOUNDS],
                 up=[b[1] for b in BOUNDS],
                 indpb=1/3)
toolbox.register("select", tools.selTournament, tournsize=3)

def run_ga(pop_size=18, ngen=5, cxpb=0.9, mutpb=0.25, hof_size=3):
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(hof_size)

    # ---- repair + evaluate INITIAL population ----
    for ind in pop:
        ind[:] = repair_and_round(ind)
    invalid = [ind for ind in pop if not ind.fitness.valid]
    for ind in invalid:
        ind.fitness.values = toolbox.evaluate(ind)
    hof.update(pop)

    # ---- safe stats key (handles rare invalids gracefully) ----
    def safe_key(ind):
        return ind.fitness.values[0] if (hasattr(ind.fitness, "values") and len(ind.fitness.values) > 0) else float("inf")

    stats = tools.Statistics(safe_key)
    stats.register("min", min)
    stats.register("avg", lambda arr: sum(arr)/len(arr))

    # ---- logging containers for later plotting ----
    hist_min, hist_avg = [], []
    all_points = []   # (pv, csp, tes, ppa, capex_musd, firm_hours)

    # log initial stats
    rec0 = stats.compile(pop)
    hist_min.append(rec0["min"]); hist_avg.append(rec0["avg"])

    # keep initial points
    for ind in pop:
        pv, csp, tes = ind
        ppa = objective(ind)[0]  # already evaluated; safe reuse
        # OPTIONAL: if you have helpers to recover CAPEX / firm hours from cache, call them here
        capex_musd = None  # fill if you expose it from evaluate_tuple
        firm_hours = None  # fill from calculate_firm_bonus if you expose
        all_points.append((pv, csp, tes, ppa, capex_musd, firm_hours))

    # ---- evolutionary loop ----
    for gen in range(ngen):
        offspring = algorithms.varAnd(pop, toolbox, cxpb=cxpb, mutpb=mutpb)

        # repair + evaluate
        for ind in offspring:
            ind[:] = repair_and_round(ind)
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = toolbox.evaluate(ind)

        # select & update HOF
        pop = toolbox.select(pop + offspring + hof.items, k=pop_size)
        hof.update(pop)

        # record stats
        rec = stats.compile(pop)
        hist_min.append(rec["min"]); hist_avg.append(rec["avg"])

        # append evaluated points
        for ind in pop:
            pv, csp, tes = ind
            ppa = ind.fitness.values[0]
            capex_musd = None
            firm_hours = None
            all_points.append((pv, csp, tes, ppa, capex_musd, firm_hours))

        print(f"Gen {gen+1:02d} | min={rec['min']:.2f} | avg={rec['avg']:.2f}")

    best = hof[0]
    print("\nBest x (desired_array_size_kWdc, P_ref_MWe, t_TES_h):",
          [round(v, 3) for v in best])
    print("Best PPA (USD/MWh):", round(best.fitness.values[0], 3))
    return best, hof, hist_min, hist_avg, all_points
if __name__ == "__main__":
    run_ga()


Gen 01 | min=176.44 | avg=183.16
Gen 02 | min=171.46 | avg=178.60
Gen 03 | min=171.46 | avg=175.28
Gen 04 | min=170.29 | avg=172.81
Gen 05 | min=167.57 | avg=170.64

Best x (desired_array_size_kWdc, P_ref_MWe, t_TES_h): [105000.0, 47, 11.5]
Best PPA (USD/MWh): 167.569
